# Choosing between key-based and keyless signing in cosign

> last_verified: 2026-09-24 · cosign (no version researched this cycle; no externally-verifiable claims made)

## Purpose

This notebook compares the two signing modes cosign supports — key-based signing with a long-lived keypair and keyless signing with a short-lived certificate bound to a workload identity — so a team can pick the mode that fits its pipeline. It covers how each mode handles key storage, signer identity, and verification, with a decision aid at the end.

## When to use

Use key-based signing when the release process needs an explicit, long-lived signing key under the team's direct control (for example, an offline release key held by a small set of maintainers). Use keyless signing when signatures are produced inside automated CI jobs where managing and rotating a shared secret is the main operational burden. Many setups mix the two: keyless for routine CI builds, key-based for milestone releases.

## Prerequisites

- A container image reference to sign (digest-based references are preferred so the signature binds to exact content).
- For key-based: a generated keypair with the private half stored securely and the public half distributed to verifiers.
- For keyless: a CI environment that can present a workload identity to an OIDC-backed issuer at signing time, plus network access to the certificate issuer and transparency log from both the signer and the verifier.
- The kit companions under `cosign/`: `configs/keyless-signing-github-actions.yaml`, `scripts/cosign-key-management-workflow.sh`, and `docs/cosign-verification-patterns.md`.


In [ ]:
# Comparison matrix: operational properties of each signing mode.
# Values are qualitative (lower/higher burden), not measured benchmarks.

MODES = {
    "key-based": {
        "credential": "long-lived keypair generated and stored by the team",
        "signer_identity": "whoever holds the private key (key must be mapped to a person/role out of band)",
        "rotation": "manual: generate, distribute, and retire keys on a schedule",
        "verifier_needs": "the public key (or its trusted location)",
        "offline_friendly": True,
    },
    "keyless": {
        "credential": "short-lived certificate issued per signing event, bound to workload identity",
        "signer_identity": "workload identity embedded in the certificate (repo, workflow, branch)",
        "rotation": "none to manage: each signature uses a fresh short-lived certificate",
        "verifier_needs": "the expected workload identity plus access to issuer and transparency-log state",
        "offline_friendly": False,
    },
}

for name, props in MODES.items():
    print(f"== {name} ==")
    for key, value in props.items():
        print(f"  {key}: {value}")

assert set(MODES) == {"key-based", "keyless"}
assert all("rotation" in p for p in MODES.values())
print("matrix OK")


## Steps

### 1. Understand what key-based signing asks of the team

Key-based signing starts with generating a keypair before anything is signed. The private half must be stored where the signing step can reach it but nothing else can — a local file with tight permissions for manual releases, or a CI secret for automated ones. Verifiers need the public half, so distributing and pinning that public key is part of the setup. The main ongoing cost is rotation: every rotation means generating a new pair, updating verifiers, and keeping the old public key around long enough to verify previously released artifacts.

### 2. Understand what keyless signing asks of the pipeline

Keyless signing removes the stored secret. At signing time the CI job authenticates as its workload identity, receives a short-lived certificate binding that identity to the signature, and the signature is recorded in a transparency log. Verification then checks two things instead of one: the cryptographic signature and that the embedded identity matches the expected workload (repository, workflow, branch). The trade-off is availability — both signing and verification depend on reaching the issuer and transparency-log services.

### 3. Compare verification side by side

- Key-based verification answers: "was this signed by the holder of the private key, checked against this public key?" Identity beyond key possession comes from how the team guards and labels its keys.
- Keyless verification answers: "was this signed on behalf of this workload identity, with a certificate and log entry to back it?" The identity is embedded, but the verifier must define the expected identity precisely (a branch-scoped rule admits any run on that branch).
- Either way, verify by digest rather than by mutable tag so the check binds to exact image content; the kit's `docs/cosign-verification-patterns.md` companion walks through the verify-gate pattern.

### 4. Apply the decision aid below

Encode the team's constraints (offline releases, tolerance for key-management overhead, availability of workload identity in CI) and let the aid recommend a mode. Mixed setups are common and legitimate: the aid flags when each side wins rather than forcing a single global answer.


In [ ]:
def recommend(needs_offline_release, has_workload_identity, wants_no_key_custody):
    """Recommend a cosign signing mode from three yes/no constraints."""
    if needs_offline_release:
        return "key-based"  # keyless needs issuer/log reachability at sign time
    if wants_no_key_custody and has_workload_identity:
        return "keyless"
    if not has_workload_identity:
        return "key-based"  # no workload identity to bind a certificate to
    return "keyless"


CASES = [
    # (offline, workload_identity, no_key_custody, expected)
    (True, True, True, "key-based"),  # air-gapped release desk
    (False, True, True, "keyless"),  # standard hosted CI pipeline
    (False, False, True, "key-based"),  # CI without workload identity
    (False, True, False, "keyless"),  # team tolerates keys but identity exists
]

for offline, identity, no_custody, expected in CASES:
    got = recommend(offline, identity, no_custody)
    status = "OK" if got == expected else "MISMATCH"
    print(f"{status}: offline={offline} identity={identity} no_custody={no_custody} -> {got}")
    assert got == expected, (offline, identity, no_custody, got)

print("decision aid OK")


## Verify

- Re-run both code cells: the matrix prints two modes with matching `rotation` entries, and the decision aid reports `OK` for all four cases.
- Cross-check the recommendation against the kit companions: the keyless CI shape in `configs/keyless-signing-github-actions.yaml` and the key-rotation workflow in `scripts/cosign-key-management-workflow.sh`.
- Confirm verification policy follows `docs/cosign-verification-patterns.md` (verify-gate) and signs/verifies by digest per `docs/container-signing-pipeline.md`.

## Common errors

- Treating the modes as mutually exclusive. A team that signs routine CI builds keylessly can still keep a long-lived key for milestone releases; document which verifier path applies to which artifact stream.
- Verifying by mutable tag. A tag can be moved after signing, so the verification step should resolve to the digest first and check the signature against that digest.
- Overly broad keyless identity rules. Accepting any run from a repository admits compromised or experimental branches; scope the expected identity to the release workflow and protected branches or tags.
- Losing the old public key after a key-based rotation. Previously released artifacts still verify against the retired key, so archive retired public keys until those artifacts are out of support.

## References

- `cosign/configs/keyless-signing-github-actions.yaml` — keyless CI wiring companion.
- `cosign/scripts/cosign-key-management-workflow.sh` — key generation and rotation companion.
- `cosign/docs/cosign-verification-patterns.md` — verify-gate patterns.
- `cosign/docs/container-signing-pipeline.md` — digest-based pipeline integration.
